In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

category_urls = [
    ("Mystery", "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html"),
    ("Historical Fiction", "https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html"),
    ("Sequential Art", "https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html")
]

raw_books = []

for cat_name, url in category_urls:
    page_url = url
    while True:
        response = requests.get(page_url)
        soup = BeautifulSoup(response.text, 'lxml')

        for book in soup.find_all('article', class_='product_pod'):
            raw_books.append({
                'title': book.h3.a['title'],
                'price': book.find('p', class_='price_color').text,
                'star_rating': book.p.get('class')[1],
                'availability': book.find('p', class_='instock availability').text.strip(),
                'category': cat_name
            })

        next_button = soup.find('li', class_='next')
        if next_button:
            next_page = next_button.a['href']
            page_url = url.rsplit('/', 1)[0] + '/' + next_page
        else:
            break

df = pd.DataFrame(raw_books)
print(f"Scraping complete! Total books scraped: {len(df)}")

Scraping complete! Total books scraped: 133


In [ ]:
df['price_gbp'] = df['price'].str.replace(r'[^\d.]', '', regex=True).astype(float)
rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
df['rating'] = df['star_rating'].map(rating_map)
df['in_stock'] = df['availability'].str.contains('In stock', case=False, na=False)
df['price_inr'] = df['price_gbp'] * 105.50
df.dropna(subset=['price_gbp', 'rating'], inplace=True)
df['rating'] = df['rating'].astype(int)
df_clean = df[['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category']].copy()
print("Cleaning complete! Here are the first 3 rows:")
display(df_clean.head(3))

Cleaning complete! Here are the first 3 rows:


,title,price_gbp,price_inr,rating,in_stock,category
0,Sharp Objects,47.82,5045.010,4,True,Mystery
1,"In a Dark, Dark Wood",19.63,2070.965,1,True,Mystery
2,The Past Never Ends,56.50,5960.750,4,True,Mystery


In [3]:
categories_df = pd.DataFrame({'category_name': df_clean['category'].unique()})
categories_df['category_id'] = range(1, len(categories_df) + 1)

books_df = pd.merge(df_clean, categories_df, left_on='category', right_on='category_name')
books_df = books_df[['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category_id']]

import sqlite3
conn = sqlite3.connect('zepto_catalogue.db')
cursor = conn.cursor()

cursor.executescript('''
    DROP TABLE IF EXISTS books;
    DROP TABLE IF EXISTS categories;

    CREATE TABLE categories (
        category_id INTEGER PRIMARY KEY,
        category_name TEXT UNIQUE
    );

    CREATE TABLE books (
        book_id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT,
        price_gbp REAL,
        price_inr REAL,
        rating INTEGER,
        in_stock INTEGER,
        category_id INTEGER,
        FOREIGN KEY (category_id) REFERENCES categories (category_id)
    );
''')

categories_df.to_sql('categories', conn, if_exists='append', index=False)
books_df.to_sql('books', conn, if_exists='append', index=False)

print("Database built and data loaded successfully!")

Database built and data loaded successfully!


In [4]:
q1 = "SELECT title, price_gbp, rating FROM books WHERE rating = 5"
display(pd.read_sql(q1, conn).head(3))

q2 = "SELECT title, price_gbp FROM books ORDER BY price_gbp DESC LIMIT 5"
display(pd.read_sql(q2, conn))

q3 = "SELECT DISTINCT rating FROM books ORDER BY rating"
display(pd.read_sql(q3, conn))

q4 = "SELECT title, price_gbp FROM books WHERE price_gbp BETWEEN 10 AND 20"
display(pd.read_sql(q4, conn).head(3))

q5 = """
SELECT c.category_name, b.title, b.rating 
FROM books b 
JOIN categories c ON b.category_id = c.category_id 
WHERE b.rating >= 4
"""
display(pd.read_sql(q5, conn).head(3))

pandas_join = pd.merge(books_df, categories_df, on='category_id')
pandas_filtered = pandas_join[pandas_join['rating'] >= 4][['category_name', 'title', 'rating']]
display(pandas_filtered.head(3))

conn.close()

,title,price_gbp,rating
0,A Time of Torment (Charlie Parker #14),48.35,5
1,What Happened on Beale Street (Secrets of the ...,25.37,5
2,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5


,title,price_gbp
0,Boar Island (Anna Pigeon #19),59.48
1,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,57.70
2,El Deafo,57.62
3,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",57.06
4,"Giant Days, Vol. 1 (Giant Days #1-4)",56.76


,rating
0,1
1,2
2,3
3,4
4,5


,title,price_gbp
0,"In a Dark, Dark Wood",19.63
1,A Murder in Time,16.64
2,That Darkness (Gardiner and Renner #1),13.92


,category_name,title,rating
0,Mystery,Sharp Objects,4
1,Mystery,The Past Never Ends,4
2,Mystery,The Murder of Roger Ackroyd (Hercule Poirot #4),4


,category_name,title,rating
0,Mystery,Sharp Objects,4
2,Mystery,The Past Never Ends,4
4,Mystery,The Murder of Roger Ackroyd (Hercule Poirot #4),4
